# Lecture 2: Variable Renewables, Electricity Demand, and Renewable Generation in Germany

This notebook introduces the temporal interaction between electricity demand and variable renewable generation in Germany. The focus is on understanding profiles, heatmaps, and residual load.

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go


In [ ]:
# Plotting options
pd.options.plotting.backend = "plotly"
template = "plotly_white"
# template = "plotly_dark"

## Investigating Renewable Generation Assets

First, we analyze wind and solar generation from single generator sources  

In [ ]:
# This block loads the wind turbine profile from a csv and plots it
df_wind = pd.read_csv("../data/windturbine_profile.csv", index_col=0, parse_dates=True)
df_wind.plot(template=template, labels={"value": "Power [MW]"})

In [ ]:
# This block loads the solar profile from a csv and plots it
df_solar = pd.read_csv("../data/pv_profile.csv", index_col=0, parse_dates=True)
df_solar.plot(template=template, color_discrete_sequence=["#EF553B"], labels={"value": "Power [W]"})

## Exploring the German Power System

To study this relationship, we use the Open Power System Data time-series dataset: [https://data.open-power-system-data.org/time_series](https://data.open-power-system-data.org/time_series)

The dataset is compiled from publicly available sources such as ENTSO-E and contains load and renewable generation profiles for different regions of the European synchronized grid. In this notebook, we focus on Germany in the year 2019.

In [ ]:
# This block loads the time series data (year 2019) for all TSOs from a csv and converts the timestamp from UTC to local time.
data = pd.read_csv("../data/time_series_60min_singleindex_filtered.csv", index_col=0, parse_dates=True)
data.index = data.index.tz_convert("Europe/Berlin") # convert timestap from UTC to local time

In [ ]:
# the following function extracts one country or TSO profile and keep only load, solar, and wind.
def tso_dataframe(df, tso):
    """Return a clean dataframe with load, solar, and aggregated wind."""
    wind_columns = [
        f"{tso}_wind_onshore_generation_actual",
        f"{tso}_wind_offshore_generation_actual",
    ]
    wind_columns = df.columns.intersection(wind_columns)   # Some regions have no offshore wind.
    df_wind = df[wind_columns].sum(axis=1).rename("wind")  # Aggregate onshore + offshore wind.

    df_solar = df[f"{tso}_solar_generation_actual"].rename("solar")
    df_load = df[f"{tso}_load_actual_entsoe_transparency"].rename("load")

    return pd.concat([df_load, df_solar, df_wind], axis=1)  # Join the three series.


## Electricity Demand and Renewable Generation in Germany

Renewable generation is weather-dependent, while electricity demand follows its own daily and seasonal patterns. This mismatch is a key challenge for power system operation.

We first explore the data using time series, heatmaps, and histograms.

In [ ]:
# read the 'Germany' dataset
df_DE = tso_dataframe(data, "DE")
# display the first few rows of the Germany dataset
df_DE.head()

In [ ]:
# Plot load together with renewable generation.
def plot_load_generation(df, **kwargs):
    """Plot load, solar, wind, and optionally residual load."""
    fig = go.Figure()
    fig.update_layout(xaxis_title="Time", yaxis_title="Power [MW]", **kwargs)
    fig.add_trace(go.Scatter(x=df.index, y=df["load"],  name="load"))
    fig.add_trace(go.Scatter(x=df.index, y=df["solar"], name="solar", stackgroup="renewables"))
    fig.add_trace(go.Scatter(x=df.index, y=df["wind"],  name="wind",  stackgroup="renewables"))
    if "residual" in df.columns:
        fig.add_trace(go.Scatter(x=df.index, y=df["residual"],  name="residual"))
    return fig


In [ ]:
# function to convert a datetime-indexed series into a day-versus-time heatmap.
def datetime_heatmap(df):
    """Create a simple heatmap from a datetime-indexed series."""
    fig = go.Figure(data=go.Heatmap(
        z=df,
        x=df.index.date,
        y=df.index.time,
        colorscale='RdBu_r'
    ))
    return fig

In [ ]:
# plot wind, solar and load for the month july 2019 in Germany
plot_load_generation(df_DE.loc["2019-07"], template=template)

In [ ]:
# plot the load as a heatmap (day versus time) for the whole year 2019 in Germany
datetime_heatmap(df_DE["load"]).update_layout(template=template, title="Power demand [MW] - Germany 2019")

In [ ]:
# plot the wind generation as a heatmap (day versus time) for the whole year 2019 in Germany
datetime_heatmap(df_DE["wind"]).update_layout(template=template, title="Wind power generation [MW]- Germany 2019")

In [ ]:
# plot the solar generation as a heatmap (day versus time) for the whole year 2019 in Germany
datetime_heatmap(df_DE["solar"]).update_layout(template=template, title="Solar power generation [MW] - Germany 2019")

## Residual Load

After looking at load, wind, and solar separately, we now combine them into one key indicator: **residual load**. It describes the remaining demand after subtracting renewable generation.

$$P_{res}(t) = P_{load}(t) - P_{solar}(t) - P_{wind}(t)$$

A positive residual load means that additional supply is still needed. A negative residual load indicates a surplus of renewable generation. The residual load duration curve helps us understand both magnitude and frequency over the year.

In [ ]:
# Add residual load to an existing dataframe.
def add_residual(df):
    """Store residual load as load minus solar and wind."""
    df["residual"] = df["load"] - df["solar"] - df["wind"]
    return df
df_DE = add_residual(df_DE)

In [ ]:
# Plot sorted annual curves for Germany.
def plot_residual_curve_DE(df, **kwargs):
    """Plot load, residual load, solar, and wind as duration curves."""
    fig = go.Figure()
    fig.update_layout(
        title="Sorted load duration curves - 2019",
        xaxis_title="Time [h]",
        yaxis_title="Power [MW]",
        width=800,
        height=600,
        **kwargs
    )
    fig.add_trace(go.Scatter(
        x=np.arange(8760),
        y=np.zeros(8760),
        line={"color": "grey", "dash": "dash"},
        opacity=0.7,
        showlegend=False
    ))
    aggregated_load = df["load"].sort_values(ascending=False).values
    aggregated_residual = df["residual"].sort_values(ascending=False).values
    aggregated_pv = df["solar"].sort_values(ascending=False).values
    aggregated_wind = df["wind"].sort_values(ascending=False).values
    time_hours = np.arange(aggregated_residual.size)
    fig.add_trace(go.Scatter(x=time_hours, y=aggregated_residual, name="Residual load", line={"color": "#19d3f3"}))
    fig.add_trace(go.Scatter(x=time_hours, y=aggregated_pv, name="Solar", line={"color": "#ff7f0e"}))
    fig.add_trace(go.Scatter(x=time_hours, y=aggregated_wind, name="Wind", line={"color": "#1f77b4"}))
    fig.add_trace(go.Scatter(x=time_hours, y=aggregated_load, name="Load", line={"color": "#d62728"}))
    return fig



<div class="alert alert-block alert-info">

**TASK 01**

Use a single scaling factor for both PV and wind generation based on the 2019 base year.

- Vary the common scaling factor with a Deepnote slider.
- Recalculate the residual load with a button press if automatic updates are not available.
- Observe how the residual load changes in the duration curve, heatmap, and time series.
- Find a setting at which the positive and negative residual energy become roughly balanced.


### Deepnote setup for the scaling study

Create the controls in Deepnote directly **above the next output cell** so that the input and the figures remain visible at the same time.

1. Create **one Slider input block**.
   - **Variable name:** `scale_factor`
   - **Minimum:** `0`
   - **Maximum:** `5`
   - **Step:** `0.1`
   - **Default value:** `1.0`

2. If automatic updates are unreliable, create a **Run button** next to the slider.
   - Use the button to rerun the output cell after changing the slider value.
   - The helper functions are placed before the output cell, so the control and the figures stay close together.

The output is split into two parts:
- first, the **residual load duration curve**
- second, a combined figure with the **heatmap** and the **time series**


In [ ]:
# function to prepare a clean dataframe for the scaling analysis.
df_plot = df_DE.copy().groupby(df_DE.index).mean(numeric_only=True)

required_cols = ["load", "solar", "wind"]
missing_cols = [c for c in required_cols if c not in df_plot.columns]
if missing_cols:
    raise ValueError(f"Missing columns in df_DE: {missing_cols}")

def infer_time_step_hours(index):
    """Return the median time step of the datetime index in hours."""
    delta_hours = pd.Series(index).sort_values().diff().dropna().dt.total_seconds() / 3600
    return float(delta_hours.median()) if not delta_hours.empty else 1.0


def compute_scaled_residual_data(df, scale_factor):
    """Scale PV and wind together and return residual load plus summary values."""
    scaled_generation = scale_factor * (df["solar"] + df["wind"])
    residual = df["load"] - scaled_generation
    residual.name = "residual_load"

    dt_hours = infer_time_step_hours(df.index)

    energy_deficit_gwh = residual.clip(lower=0).sum() * dt_hours / 1000
    energy_surplus_gwh = (-residual.clip(upper=0)).sum() * dt_hours / 1000

    summary = {
        "min_mw": residual.min(),
        "max_mw": residual.max(),
        "mean_mw": residual.mean(),
        "energy_deficit_gwh": energy_deficit_gwh,
        "energy_surplus_gwh": energy_surplus_gwh,
    }

    return residual, summary

def build_duration_curve_figure(residual, scale_factor, template="plotly_white"):
    """Create the annual duration curve of the residual load."""
    duration_curve = residual.dropna().sort_values(ascending=False)
    x = np.arange(1, len(duration_curve) + 1)

    fig = go.Figure()
    fig.add_trace(
        go.Scatter(
            x=x,
            y=duration_curve.values,
            mode="lines",
            name="Duration curve",
            hovertemplate="Sorted hour: %{x}<br>Residual load: %{y:.2f} MW<extra></extra>",
        )
    )

    fig.add_hline(y=0, line_dash="dash", line_width=1)

    fig.update_layout(
        template=template,
        title=f"Residual load duration curve (common scaling factor = {scale_factor:.2f})",
        xaxis_title="Sorted hours",
        yaxis_title="Residual load [MW]",
        height=450,
        showlegend=False,
    )

    return fig

def build_timeseries_figure(residual, scale_factor, template="plotly_white"):
    """Create the time series plot of the residual load."""
    fig = go.Figure()

    fig.add_trace(
        go.Scatter(
            x=residual.index,
            y=residual.values,
            mode="lines",
            name="Residual load",
            hovertemplate="Time: %{x}<br>Residual load: %{y:.2f} MW<extra></extra>",
        )
    )

    fig.add_hline(y=0, line_dash="dash", line_width=1)

    fig.update_layout(
        template=template,
        title=f"Residual load time series (common scaling factor = {scale_factor:.2f})",
        xaxis_title="Time",
        yaxis_title="Residual load [MW]",
        height=450,
        showlegend=False,
    )

    return fig

In [ ]:
scale_factor = 1

In [ ]:
# Deepnote input block:
# Create one slider named "scale_factor" above this cell.
# Example range: 0.0 to 5.0 with step size 0.1

try:
    scale_factor
except NameError:
    scale_factor = 1.0  # fallback if no Deepnote input block is defined

template = "plotly_white"

residual, summary = compute_scaled_residual_data(df_plot, scale_factor)

print(f"Common scaling factor: {scale_factor:.2f}")
print(f"Minimum residual load: {summary['min_mw']:.2f} MW")
print(f"Maximum residual load: {summary['max_mw']:.2f} MW")
print(f"Mean residual load: {summary['mean_mw']:.2f} MW")
print(f"Balanced energy deficit: {summary['energy_deficit_gwh']:.2f} GWh")
print(f"Balanced energy surplus: {summary['energy_surplus_gwh']:.2f} GWh")

fig_duration = build_duration_curve_figure(residual, scale_factor, template=template)
fig_timeseries = build_timeseries_figure(residual, scale_factor, template=template)

fig_duration.show()
fig_timeseries.show()

## Electricity Demand and Renewable Generation in Germany at Regional Level

<img src="https://upload.wikimedia.org/wikipedia/commons/8/82/Regelzonen_mit_%C3%9Cbertragungsnetzbetreiber_in_Deutschland.png" align="left" width="400" alt="German TSO map">

The figure on the left shows the four German transmission system operators (TSOs):

- Amprion
- TenneT
- TransnetBW
- 50Hertz

Each TSO is responsible for operating and securing the transmission grid in its region. By disaggregating the national data, we can compare how regional demand and renewable generation patterns differ across Germany.


In [ ]:
# create a dictionary with a dataframe to each TSO
df_tso = {tso: tso_dataframe(data, tso) for tso in ("DE_tennet", "DE_50hertz", "DE_amprion", "DE_transnetbw")}
df_tso["Germany"] = df_DE # also include the Germany-wide aggregated data
# calculate the residual to all TSOs
df_tso = {tso: add_residual(df) for tso, df in df_tso.items()}

In [ ]:
# plot 50hertz's demand and generation in July 2019 as an example
plot_load_generation(df_tso["DE_50hertz"].loc["2019-07"], template=template)

In [ ]:
# function to plot the residual load duration curve for the TSOs
def plot_residual_curve(profiles, year="2019", **kwargs):
    fig = go.Figure()
    fig.update_layout(title="Residual load duration curve - "+year, xaxis_title="Time [h]", yaxis_title="Power [MW]", width=800, height=600, **kwargs)
    fig.add_trace(go.Scatter(x=np.arange(8760), y=np.zeros(8760), line={"color": "grey", "dash": "dash"}, opacity=0.7, showlegend=False, name=0))
    # fig.update_yaxes(exponentformat="power")
    for tso, df in profiles.items():
        aggregated_residual = df.loc[year, "residual"].sort_values(ascending=False).values
        time_hours = np.arange(aggregated_residual.size)
        fig.add_trace(go.Scatter(x=time_hours, y=aggregated_residual, name=tso))
    return fig

In [ ]:
# function to compare residual load duration curves across several regions.
def plot_residual_curve(profiles, year="2019", **kwargs):
    """Plot one residual load duration curve per region."""
    fig = go.Figure()
    fig.update_layout(
        title="Residual load duration curve - " + year,
        xaxis_title="Time [h]",
        yaxis_title="Power [MW]",
        width=800,
        height=600,
        **kwargs
    )
    fig.add_trace(go.Scatter(
        x=np.arange(8760),
        y=np.zeros(8760),
        line={"color": "grey", "dash": "dash"},
        opacity=0.7,
        showlegend=False,
        name=0
    ))
    for tso, df in profiles.items():
        aggregated_residual = df.loc[year, "residual"].sort_values(ascending=False).values
        time_hours = np.arange(aggregated_residual.size)
        fig.add_trace(go.Scatter(x=time_hours, y=aggregated_residual, name=tso))
    return fig


In [ ]:
# Plot the residual load duration curve for all TSOs.
plot_residual_curve(df_tso, template=template)